In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.6`

import $ivy.$

In [2]:
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.OFF)

import org.apache.log4j.{Level, Logger}

In [3]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import java.sql.Timestamp
import java.time._

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import java.sql.Timestamp
import java.time._

In [4]:
val spark = SparkSession
                .builder()
                .master("local[*]")
                .appName("Hello Spark")
                .config("spark.log.level", "WARN")
                .getOrCreate()

import spark.implicits._

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/03/13 13:57:35 INFO SparkContext: Running Spark version 3.5.5
25/03/13 13:57:35 INFO SparkContext: OS info Mac OS X, 15.3.2, aarch64
25/03/13 13:57:35 INFO SparkContext: Java version 11.0.26
25/03/13 13:57:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


spark: SparkSession = org.apache.spark.sql.SparkSession@2ee43b6c
import spark.implicits._

In [5]:
println(s"spark.version == ${spark.version}")

spark.version == 3.5.5


In [6]:
val df = List(
      (1L, 2.0, "string1", LocalDate.of(2000, 1, 1), Timestamp.valueOf("2023-01-01 12:00:00")),
      (2L, 3.0, "string2", LocalDate.of(2000, 2, 1), Timestamp.valueOf("2023-02-01 12:00:00")),
      (3L, 4.0, "string3", LocalDate.of(2000, 3, 1), Timestamp.valueOf("2023-03-01 12:00:00"))
    ).toDF("a", "b", "c", "d", "e")

df: org.apache.spark.sql.package.DataFrame = [a: bigint, b: double ... 3 more fields]

In [7]:
df.printSchema()

root
 |-- a: long (nullable = false)
 |-- b: double (nullable = false)
 |-- c: string (nullable = true)
 |-- d: date (nullable = true)
 |-- e: timestamp (nullable = true)



In [8]:
df.show()

+---+---+-------+----------+-------------------+
|  a|  b|      c|         d|                  e|
+---+---+-------+----------+-------------------+
|  1|2.0|string1|2000-01-01|2023-01-01 12:00:00|
|  2|3.0|string2|2000-02-01|2023-02-01 12:00:00|
|  3|4.0|string3|2000-03-01|2023-03-01 12:00:00|
+---+---+-------+----------+-------------------+



In [9]:
df.withColumn("upper_c", upper($"c")).show()

+---+---+-------+----------+-------------------+-------+
|  a|  b|      c|         d|                  e|upper_c|
+---+---+-------+----------+-------------------+-------+
|  1|2.0|string1|2000-01-01|2023-01-01 12:00:00|STRING1|
|  2|3.0|string2|2000-02-01|2023-02-01 12:00:00|STRING2|
|  3|4.0|string3|2000-03-01|2023-03-01 12:00:00|STRING3|
+---+---+-------+----------+-------------------+-------+



In [10]:
df.select(col("c")).show()

+-------+
|      c|
+-------+
|string1|
|string2|
|string3|
+-------+



In [11]:
df.filter($"a" === 1L).show()

+---+---+-------+----------+-------------------+
|  a|  b|      c|         d|                  e|
+---+---+-------+----------+-------------------+
|  1|2.0|string1|2000-01-01|2023-01-01 12:00:00|
+---+---+-------+----------+-------------------+



In [12]:
val df0 = List((1L, "100000"), (2L, "2000"), (3L, "3000")).toDF("id", "exampleCol")
val df = df0.select($"id", col("exampleCol").cast("decimal").alias("exampleColDecimal"))

df0: org.apache.spark.sql.package.DataFrame = [id: bigint, exampleCol: string]
df: org.apache.spark.sql.package.DataFrame = [id: bigint, exampleColDecimal: decimal(10,0)]

In [13]:
df.printSchema

root
 |-- id: long (nullable = false)
 |-- exampleColDecimal: decimal(10,0) (nullable = true)



In [14]:
df.filter($"exampleCol" === 100000).show

+---+-----------------+
| id|exampleColDecimal|
+---+-----------------+
|  1|           100000|
+---+-----------------+



In [15]:
df.filter(col("exampleCol") === 100000).show

+---+-----------------+
| id|exampleColDecimal|
+---+-----------------+
|  1|           100000|
+---+-----------------+



In [16]:
df.filter(col("exampleCol") === 100000).explain("extended")

== Parsed Logical Plan ==
'Filter ('exampleCol = 100000)
+- Project [id#113L, cast(exampleCol#114 as decimal(10,0)) AS exampleColDecimal#117]
   +- Project [_1#108L AS id#113L, _2#109 AS exampleCol#114]
      +- LocalRelation [_1#108L, _2#109]

== Analyzed Logical Plan ==
id: bigint, exampleColDecimal: decimal(10,0)
Project [id#113L, exampleColDecimal#117]
+- Filter (cast(exampleCol#114 as int) = 100000)
   +- Project [id#113L, cast(exampleCol#114 as decimal(10,0)) AS exampleColDecimal#117, exampleCol#114]
      +- Project [_1#108L AS id#113L, _2#109 AS exampleCol#114]
         +- LocalRelation [_1#108L, _2#109]

== Optimized Logical Plan ==
LocalRelation [id#113L, exampleColDecimal#117]

== Physical Plan ==
LocalTableScan [id#113L, exampleColDecimal#117]



In [17]:
val df2 = df0.withColumn("exampleColDecimal", col("exampleCol").cast("decimal")).drop("exampleCol")

df2: org.apache.spark.sql.package.DataFrame = [id: bigint, exampleColDecimal: decimal(10,0)]

In [18]:
df2.printSchema

root
 |-- id: long (nullable = false)
 |-- exampleColDecimal: decimal(10,0) (nullable = true)



In [19]:
df2.filter($"exampleCol" === 100000).explain("extended")

== Parsed Logical Plan ==
'Filter ('exampleCol = 100000)
+- Project [id#113L, exampleColDecimal#138]
   +- Project [id#113L, exampleCol#114, cast(exampleCol#114 as decimal(10,0)) AS exampleColDecimal#138]
      +- Project [_1#108L AS id#113L, _2#109 AS exampleCol#114]
         +- LocalRelation [_1#108L, _2#109]

== Analyzed Logical Plan ==
id: bigint, exampleColDecimal: decimal(10,0)
Project [id#113L, exampleColDecimal#138]
+- Filter (cast(exampleCol#114 as int) = 100000)
   +- Project [id#113L, exampleColDecimal#138, exampleCol#114]
      +- Project [id#113L, exampleCol#114, cast(exampleCol#114 as decimal(10,0)) AS exampleColDecimal#138]
         +- Project [_1#108L AS id#113L, _2#109 AS exampleCol#114]
            +- LocalRelation [_1#108L, _2#109]

== Optimized Logical Plan ==
LocalRelation [id#113L, exampleColDecimal#138]

== Physical Plan ==
LocalTableScan [id#113L, exampleColDecimal#138]

